# Kaggle: Embed VN Legal Chunks với bge-m3

**Workflow** (per Decision 16):
1. Upload `chunks.jsonl` vào Kaggle Dataset (private).
2. Run notebook này trên Kaggle với GPU T4 enabled.
3. Download output zip → unzip vào `data/chroma_db/` ở local repo.

**Estimate**: 42k chunks × bge-m3 trên T4 ≈ 10–15 phút.

**Setup Kaggle**:
- Settings → Accelerator: GPU T4 x2 (hoặc P100).
- Internet: ON (cần download model).
- Add Data: dataset chứa `chunks.jsonl`.

In [ ]:
# Cell 1 — Install deps
!pip install -q sentence-transformers==3.3.1 chromadb==0.6.3

In [ ]:
# Cell 2 — Imports + config
import json
import os
import time
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "nhonhoccode/zalo-legal-bge-m3-finetuned"
BATCH_SIZE = 64  # T4 fits 64; nếu OOM giảm xuống 32

# Kaggle paths
INPUT_PATH = Path("/kaggle/input/datasets/nhondangcode/nlp-2026")  # tìm chunks.jsonl trong subfolder
OUTPUT_DIR = Path("/kaggle/working/chroma_db")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect chunks file
chunks_files = list(INPUT_PATH.rglob("chunks.jsonl"))
assert chunks_files, "Upload chunks.jsonl vào Kaggle Dataset trước!"
CHUNKS_FILE = chunks_files[0]

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"Input:  {CHUNKS_FILE}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Cell 3 — Load chunks
chunks = []
with CHUNKS_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

print(f"Loaded {len(chunks)} chunks")
print("Sample:", chunks[0])

In [ ]:
# Cell 4 — Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Loaded {MODEL_NAME} on {device}, dim={model.get_sentence_embedding_dimension()}")

In [ ]:
# Cell 5 — Encode
texts = [c["text"] for c in chunks]
ids = [c["chunk_id"] for c in chunks]

t0 = time.time()
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
elapsed = time.time() - t0
print(f"Encoded {len(texts)} chunks in {elapsed:.1f}s ({len(texts)/elapsed:.1f}/s)")
print(f"Shape: {embeddings.shape}, dtype: {embeddings.dtype}")

In [ ]:
# Cell 6 — Save numpy backup
np.save("/kaggle/working/embeddings.npy", embeddings.astype(np.float32))
with open("/kaggle/working/chunk_ids.json", "w") as f:
    json.dump(ids, f)
print("Saved /kaggle/working/embeddings.npy + chunk_ids.json")

In [ ]:
# Cell 7 — Save vào ChromaDB persistent
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(path=str(OUTPUT_DIR), settings=Settings(anonymized_telemetry=False))
try:
    client.delete_collection("legal_vn")
except Exception:
    pass
collection = client.create_collection(name="legal_vn", metadata={"hnsw:space": "cosine"})

# Add in batches để tránh OOM
BATCH_ADD = 500
for i in range(0, len(chunks), BATCH_ADD):
    batch = chunks[i:i+BATCH_ADD]
    batch_emb = embeddings[i:i+BATCH_ADD].tolist()
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        documents=[c["text"] for c in batch],
        embeddings=batch_emb,
        metadatas=[
            {
                "law_id":     str(c.get("law_id") or ""),
                "law_title":  str(c.get("law_title") or "")[:200],
                "article_id": str(c.get("article_id") or ""),
                "domain":     str(c.get("domain") or ""),
                # Skip None values: ChromaDB rejects None in metadata
                **{
                    k: v for k, v in (c.get("metadata") or {}).items()
                    if v is not None and isinstance(v, (str, int, float, bool))
                },
            }
            for c in batch
        ],
    )

print(f"ChromaDB: {collection.count()} records persisted to {OUTPUT_DIR}")

In [ ]:
# Cell 8 — Sanity check + zip output
import shutil

# Test query
q_text = "Người lao động bị sa thải trái pháp luật được bồi thường gì?"
q_vec = model.encode([q_text], normalize_embeddings=True)[0].tolist()
results = collection.query(query_embeddings=[q_vec], n_results=3)
for i, doc in enumerate(results["documents"][0]):
    print(f"--- top {i+1} ---")
    print(doc[:200])
    print("meta:", results["metadatas"][0][i])
    print()

# Zip cho dễ download
shutil.make_archive("/kaggle/working/chroma_db", "zip", OUTPUT_DIR)
print("\nDownload: /kaggle/working/chroma_db.zip + /kaggle/working/embeddings.npy")